# DevMind AI — Data Cleaning, Normalization & Splitting

This notebook processes approved raw pilot datasets into a unified JSONL format. It does not assume all datasets have the same schema. You must map each source's fields after inspection.

In [ ]:
!pip -q install pandas scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, hashlib, re, pandas as pd
from sklearn.model_selection import train_test_split
BASE = Path('/content/drive/MyDrive/DevMindAI/datasets')
RAW = BASE / 'raw'; PROCESSED = BASE / 'processed'; PROCESSED.mkdir(parents=True, exist_ok=True)
print('Raw files:', list(RAW.glob('*.jsonl')))

## 1. Define a source-specific mapper
Each dataset must be mapped into these fields: `instruction`, `input`, `response`, `language`, `source_dataset`, `source_record_id`. Do not invent missing answers.

In [ ]:
def normalize_record(row, source_dataset, record_id):
    # TODO: Edit these mappings for the dataset currently being processed.
    instruction = row.get('instruction') or row.get('question') or row.get('prompt') or ''
    input_text = row.get('input') or row.get('code') or row.get('problem') or ''
    response = row.get('output') or row.get('answer') or row.get('solution') or row.get('completion') or ''
    language = row.get('language') or row.get('lang') or 'unknown'
    return {'instruction': str(instruction).strip(), 'input': str(input_text).strip(),
            'response': str(response).strip(), 'language': str(language).strip().lower(),
            'source_dataset': source_dataset, 'source_record_id': str(record_id)}

def is_valid(record):
    text = ' '.join([record['instruction'], record['input'], record['response']]).strip()
    if len(text) < 40: return False
    if not record['response']: return False
    if record['instruction'] == record['response']: return False
    return True


In [ ]:
# Load JSONL files and normalize them. Review/mapping may be required per source.
records = []
for file in RAW.glob('*.jsonl'):
    source = file.stem
    with file.open(encoding='utf-8') as f:
        for i, line in enumerate(f):
            try:
                row = json.loads(line)
                rec = normalize_record(row, source, i)
                if is_valid(rec): records.append(rec)
            except Exception as e:
                print(f'Skipped {file.name} line {i}:', e)
df = pd.DataFrame(records)
print('Normalized records:', len(df))
display(df.head(5))

## 2. Exact deduplication
We use a stable hash of normalized instruction/input/response text. Near-duplicate detection will be a later, separate step.

In [ ]:
def text_hash(row):
    text = '\\n'.join([row['instruction'], row['input'], row['response']]).lower()
    text = re.sub(r'\\s+', ' ', text).strip()
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

if len(df):
    df['hash'] = df.apply(text_hash, axis=1)
    before = len(df)
    df = df.drop_duplicates(subset='hash').drop(columns='hash').reset_index(drop=True)
    print('Removed exact duplicates:', before-len(df))
else:
    print('No records yet. Download approved pilot datasets first.')

## 3. Quality review and split
Random splitting is a first baseline. For datasets containing related problems, future work should use source/project-aware splits to reduce leakage.

In [ ]:
if len(df) >= 10:
    train, temp = train_test_split(df, test_size=0.20, random_state=42)
    valid, test = train_test_split(temp, test_size=0.50, random_state=42)
    splits = {'train': train, 'validation': valid, 'test': test}
    for name, part in splits.items():
        path = PROCESSED / f'{name}.jsonl'
        part.to_json(path, orient='records', lines=True, force_ascii=False)
        print(name, len(part), path)
else:
    print('Need at least 10 valid records for this pilot split. Current:', len(df))

## 4. Dataset report
Review counts and source distribution before training.

In [ ]:
if len(df):
    display(df.groupby(['source_dataset','language'], dropna=False).size().reset_index(name='count'))
    report = {'total_after_cleaning': int(len(df)), 'sources': df['source_dataset'].value_counts().to_dict(),
              'languages': df['language'].value_counts().to_dict()}
    (PROCESSED / 'dataset_report.json').write_text(json.dumps(report, indent=2))
    print('Report saved:', PROCESSED / 'dataset_report.json')

## Next
After dataset review and cleaning, we will train a tokenizer and create the first small model configuration. Training from scratch on Colab free GPU will be an educational baseline, not a production-level foundation model.